# House Price Prediction
**Objective:** Predict house prices from features like size, location and number of rooms.

**Dataset:** California Housing (built into scikit-learn). The Boston Housing dataset was removed from scikit-learn (v1.2+) because of ethical concerns, so this is the standard replacement.

**Steps:** Load & explore → preprocess → split → train → evaluate

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

## 2. Load the dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df.rename(columns={"MedHouseVal": "Price"}, inplace=True)  # target, in units of $100,000

print("Shape:", df.shape)
df.head()

**Column meanings**
- `MedInc` – median income in the block
- `HouseAge` – median house age
- `AveRooms` / `AveBedrms` – average rooms / bedrooms per household
- `Population`, `AveOccup` – population and average occupancy
- `Latitude`, `Longitude` – location
- `Price` – median house value (target, ×$100,000)

## 3. Explore the data

In [ ]:
df.info()
print("\nMissing values:\n", df.isnull().sum())
df.describe().T

In [ ]:
df.hist(bins=40, figsize=(14, 10), edgecolor="black")
plt.suptitle("Feature distributions", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="Longitude", y="Latitude", hue="Price",
                palette="viridis", size="Population", sizes=(5, 80), alpha=0.5)
plt.title("House prices by location")
plt.show()

## 4. Preprocessing
The dataset has no missing values, but the imputer is included so the same pipeline works on other housing datasets. Scaling (normalization) is done inside a pipeline so test data never leaks into training.

In [ ]:
# Optional: cap extreme prices (the dataset clips values at 5.0)
df = df[df["Price"] < 5.0]

X = df.drop("Price", axis=1)
y = df["Price"]

## 5. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 6. Train a Linear Regression model

In [ ]:
lin_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

lin_model.fit(X_train, y_train)
y_pred_lin = lin_model.predict(X_test)

## 7. Evaluate

In [ ]:
def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "Model": name,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

results = [evaluate("Linear Regression", y_test, y_pred_lin)]
pd.DataFrame(results)

## 8. (Bonus) Compare with Random Forest

In [ ]:
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

results.append(evaluate("Random Forest", y_test, y_pred_rf))
pd.DataFrame(results).round(4)

## 9. Visualize predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, pred, title in zip(axes, [y_pred_lin, y_pred_rf],
                           ["Linear Regression", "Random Forest"]):
    ax.scatter(y_test, pred, alpha=0.3)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    ax.set_xlabel("Actual price")
    ax.set_ylabel("Predicted price")
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test - y_pred_rf
plt.figure(figsize=(8, 4))
sns.histplot(residuals, bins=40, kde=True)
plt.title("Residuals (Random Forest)")
plt.xlabel("Actual - Predicted")
plt.show()

## 10. Which features matter most?

In [ ]:
coef = pd.Series(lin_model.named_steps["model"].coef_, index=X.columns).sort_values()
imp = pd.Series(rf_model.named_steps["model"].feature_importances_, index=X.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
coef.plot(kind="barh", ax=axes[0], title="Linear Regression coefficients")
imp.plot(kind="barh", ax=axes[1], title="Random Forest feature importance")
plt.tight_layout()
plt.show()

## 11. Predict for a new house

In [ ]:
new_house = pd.DataFrame([{
    "MedInc": 5.0, "HouseAge": 20, "AveRooms": 6.0, "AveBedrms": 1.0,
    "Population": 1500, "AveOccup": 3.0, "Latitude": 34.05, "Longitude": -118.25
}])

pred = rf_model.predict(new_house)[0]
print(f"Predicted price: ${pred * 100000:,.0f}")

## 12. Save the model

In [ ]:
joblib.dump(rf_model, "house_price_model.pkl")
print("Model saved as house_price_model.pkl")

## Conclusion
- Linear Regression gives a simple baseline; Random Forest usually performs better because it captures non-linear relationships.
- Median income and location (latitude/longitude) are the strongest price drivers.
- **Next ideas:** feature engineering (e.g. rooms per person), hyperparameter tuning with `GridSearchCV`, trying Gradient Boosting / XGBoost, and wrapping the model in a Streamlit app.